# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aneeqahabib/FlyRank_ML_Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Ranking Signal Analysis**

Signal analysis is the foundation of all search and content work. Before building any model or ranking system, we need to understand which observable measurements — position tier, CTR, content age, impressions, engagement patterns, etc. — actually associate with performance outcomes. This lane answers a prior question: "Which signals should we even pay attention to?" By auditing signals honestly, we discover what the data actually shows (not what we assume), establish which measurements are worth engineering effort, and build transparent baselines that future work can beat. This is the essential first step: core idea first, modeling second.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### The Research Question

**Which safe, observable content and search signals are most strongly associated with visibility, clicks, engagement, or performance movement?**

This is a signal audit, not a ranking system. We want to identify which metrics deserve focus.

### The Decision

Strategy teams and content leaders decide where to invest effort. Should they focus on:
- Content freshness (days since last update)?
- Current position tier (which search results pages pages rank on)?
- Search volume around the topic?
- Current engagement (CTR, scroll rate, session depth)?
- Content length or structure?

Without a signal audit, teams optimize based on intuition, guesswork, or outdated product rules. With one, they make strategy choices backed by data.

### Who Acts On It

- **Strategy teams** deciding where to build new content or expand existing areas
- **Editorial leads** choosing which pages deserve refresh investment
- **SEO specialists** prioritizing optimization efforts
- **Product managers** designing content performance dashboards and alerts

### The Cost of Wrong Choices

| Wrong Signal Choice | Cost |
|---|---|
| Over-emphasize search volume | Teams optimize for keyword demand that doesn't translate to actual traffic or engagement |
| Ignore position/visibility | Miss that ranking position is the dominant factor in CTR; waste effort elsewhere |
| Focus only on engagement | Ignore that some pages are simply not getting impressions (visibility problem, not content problem) |
| Trust a noisy signal | Make decisions on low-volume wiggles; waste effort on noise, miss real patterns |

### Why Data Helps

**Observational data cannot prove causation**, but it can reveal associations. A signal audit shows which metrics co-vary with outcomes. This informs strategy — it doesn't automate ranking or guarantee outcomes, but it prevents teams from optimizing based on factors that don't actually correlate with what they care about.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [13]:
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queu

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print("Dataset Overview")
print("="*70)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"\nDate snapshot: This data aligns with warehouse build 2026-07-03 (daily facts through 2026-06-30)")
print(f"Grain: One row per content item (deduplicated by content_id)")

print("\n" + "="*70)
print("SIGNAL AUDIT: Which signals show variation and association?")
print("="*70)

# SIGNAL 1: Position Tier — Does ranking position associate with clicks?
print("\n[SIGNAL 1] Position Tier (ranking visibility)")
print("-"*70)
print("Question: Do pages at higher search positions get more clicks?")
print()

# Filter to pages with meaningful impression volume (to avoid noise)
visible_pages = df[df['impressions_90d'] >= 100]
print(f"Filtered to {len(visible_pages):,} pages with impressions >= 100 (noise floor)\n")

pos_ctr = visible_pages.groupby('position_tier')['ctr'].agg(['mean', 'std', 'count']).round(4)
print("Mean CTR by position tier:")
print(pos_ctr)

max_ctr = pos_ctr['mean'].max()
min_ctr = pos_ctr['mean'].min()
ratio = max_ctr / min_ctr if min_ctr > 0 else np.inf

print(f"\n STRONG SIGNAL: Position tier shows {ratio:.1f}x variation in CTR")
print(f"  - Pages ranking in position 1-3 (top_3) avg {pos_ctr.loc['top_3', 'mean']:.4f} CTR")
print(f"  - Pages ranking in deep results (deep) avg {pos_ctr.loc['deep', 'mean']:.4f} CTR")
print(f"  Signal interpretation: Position tier shows the strongest observed association with CTR among the signals examined.")
print(f"  Decision implication: Consider position separately from content quality.")

# SIGNAL 2: Search Volume — Does keyword demand predict page impressions?
print("\n[SIGNAL 2] Search Volume (keyword demand)")
print("-"*70)
print("Question: Do pages targeting high-volume keywords get more impressions?")
print()

corr_sv = df['search_volume'].corr(df['impressions_90d'])
print(f"Pearson correlation (search_volume ↔ impressions_90d): {corr_sv:.4f}")
print(f"\n WEAK SIGNAL: Correlation near zero. Search volume barely predicts impressions.")
print(f"  Implication: High keyword demand ≠ guaranteed high page impressions")
print(f"  In this dataset, keyword search volume alone shows almost no linear relationship with page impressions.")
print(f"  Decision implication: Optimize for position/relevance, not just volume.")

# SIGNAL 3: Content Age and Performance Trend
print("\n[SIGNAL 3] Content Age (freshness context)")
print("-"*70)
print("Question: Do older pages show different performance trends?")
print()

age_by_trend = df.groupby('trend_direction')['content_age_days'].agg([
    ('median', 'median'),
    ('mean', 'mean'),
    ('count', 'count')
]).round(0)

print("Median content age by trend direction:")
for trend in age_by_trend.index:
    median_age = int(age_by_trend.loc[trend, 'median'])
    count = int(age_by_trend.loc[trend, 'count'])
    print(f"  {trend:8s}: {median_age:5d} days ({count:,} pages)")

print(f"\n MODERATE SIGNAL: Content age varies across performance trends")
print(f"  - Declining pages: median {int(age_by_trend.loc['down', 'median'])} days old")
print(f"  - Growing pages:   median {int(age_by_trend.loc['up', 'median'])} days old")
print(f"""  Implication: Content age alone does not explain performance changes. Older pages
      in this dataset are not necessarily declining and should be evaluated alongside other
      quality and relevance signals.""")
print(f"""  Decision implication: Use age to prioritize reviews, but do not treat it as a
standalone indicator of content health.""")

print("\n" + "="*70)
print("SUMMARY: Why This Audit Matters")
print("="*70)
print(f"""
Three signals, three lessons:

1. POSITION IS KING (6x variation in CTR)
   → Visibility dominates clicks. A page's ranking position predicts its CTR far better than
     other factors. Strategy implication: don't confuse content quality with search position.

2. SEARCH VOLUME IS DECOUPLED (~0 correlation)
   → Keyword demand does NOT automatically translate to page impressions. This is the
     surprising finding that breaks the intuitive "chase volume" strategy. Implication: your
     ranking and relevance matter more than keyword hotness.

3. AGE SIGNALS CONTEXT (age provides context, not causation)
→ Content age should be interpreted as contextual rather than causal. In this dataset, older
  pages were more often stable or growing than declining, indicating that freshness alone
  does not determine performance. Implication: use age to help prioritize reviews, but base
  optimization decisions on multiple signals rather than age alone.

These three findings set the stage for deeper analysis: we now know which signals to trust,
which to be skeptical of, and where human judgment remains essential.
""")



Dataset Overview
Rows: 30,000
Columns: 44

Date snapshot: This data aligns with warehouse build 2026-07-03 (daily facts through 2026-06-30)
Grain: One row per content item (deduplicated by content_id)

SIGNAL AUDIT: Which signals show variation and association?

[SIGNAL 1] Position Tier (ranking visibility)
----------------------------------------------------------------------
Question: Do pages at higher search positions get more clicks?

Filtered to 22,006 pages with impressions >= 100 (noise floor)

Mean CTR by position tier:
                 mean     std  count
position_tier                       
deep           0.0554  0.1699    879
page_1         0.3548  0.5023   8633
page_3_5       0.1424  0.2290   6058
striking       0.2558  0.3467   5903
top_3          0.3341  0.4875    533

 STRONG SIGNAL: Position tier shows 6.4x variation in CTR
  - Pages ranking in position 1-3 (top_3) avg 0.3341 CTR
  - Pages ranking in deep results (deep) avg 0.0554 CTR
  Signal interpretation: Position 

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful Words: What I Can and Can't Claim

### What I Can Claim (Observed, Directional, Decision-Support)

This analysis is based only on **observable signals** from the dataset, including:

- Impressions, clicks, CTR, and position
- Content metadata (age, word count, content type)
- Engagement metrics
- Derived metrics (position tier, age tier, trend indicators)

I can make claims such as:

### ✓ Observed Associations

- "Position tier shows a strong association with CTR in this dataset."
- "Search volume has almost no linear relationship with impressions in this dataset."
- "Content age varies across different performance trends."

### ✓ Directional Insights

- "The data suggests that position is a stronger signal for CTR variation than search volume."
- "Content age provides context for prioritizing reviews but does not determine performance."

### ✓ Decision Support

- "These findings provide evidence to help teams prioritize investigation and optimization opportunities."

---

## What I Cannot Claim

This analysis **does not prove causation** and does not reveal search engine algorithms.

### ✗ Causal Claims

Cannot say:

> "Position causes higher CTR."

Instead:

> "Higher-ranking pages tend to have higher CTR."

Cannot say:

> "Old content causes traffic decline."

Instead:

> "Content age is associated with different performance patterns."

---

### ✗ Google Algorithm Claims

Cannot say:

- "These are Google's ranking factors."
- "This explains Google's algorithm."
- "This predicts how Google ranks pages."

Reason: We observe content performance **after ranking has occurred**, not the factors used by Google's ranking system.

---

### ✗ Guaranteed Outcomes

Cannot say:

- "Updating this page will definitely recover traffic."
- "This signal guarantees a problem."

Instead:

- "This page may be a candidate for review based on observed signals."

---

## Language Rules

| ✓ Use | ✗ Avoid |
|---|---|
| "The data suggests..." | "This proves..." |
| "We observe..." | "This causes..." |
| "Associated with..." | "Drives..." |
| "In this dataset..." | "Always..." |
| "May indicate..." | "Guarantees..." |

---

## Key Principle

## Key Principle

**Lane 1 is a signal audit, not a ranking model.**

The goal is to identify which safe content and search signals are associated with performance outcomes such as visibility, clicks, engagement, and movement. The analysis helps teams understand which signals show meaningful relationships, which assumptions are supported or challenged by the data, and where to focus further investigation.

It does **not** build a ranking system, automate decisions, predict Google behavior, or establish causal relationships.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.